# From the coursework to a model factory — a CSED 504 mini-capstone
### one training loop, made **fast on every machine**, for **CNNs *and* Transformers**, up to **1.28M images**

Across **501 → 502 → 503** we learned these models *by hand*. Then we spent **months** making them
actually **train** — fast, correctly, on every machine we could get (**Mac / Windows / Linux / Colab**),
and at a scale the coursework never touched (**ImageNet-32: 1.28M images**). This notebook is the receipt.

It runs top to bottom and:
1. rebuilds the training loop we first wrote by hand in 502, then **measures** the tricks (and the traps)
   that made it fast — the *rewrites*, with numbers and graphs;
2. **trains real models** (ResNet-18 and a ViT) with a live **dashboard**;
3. scales up to **ImageNet-32** to show *where the CNN breaks down and the Transformer becomes the win*;
4. **predicts** a big run's cost from small benchmarks, then checks the prediction against reality.

### The ladder: how much of the training do *you* write?
- **501 — classical ML (`sklearn`/`pandas`).** `model.fit()` *is* the training — no loop, no device.
- **502 — drop to the metal.** A hand-written **`Solver`**, ConvNets from scratch, then the PyTorch intro
  and hand-built attention for captioning. **You** write the loop.
- **503 — industrialize it.** Self-attention from scratch (minGPT), the HF `Trainer`, and the first
  cross-platform sweep (`hyper_sweep.py` + `cuda_check.py` — the ancestor of `src/common/gpu_check.py`).

`sklearn.fit()` → write the loop → industrialize it. Two rungs were only *just* reached: **GPU training**
(stops at `.to(device)`; we never wrote mixed precision) and **Vision Transformers** (we did attention for
*text*, never for image patches). This notebook adds the next rung — and shows the scars.

## ⚙️ Configuration — `fast` vs `full`

**Read this cell first.** Everything downstream scales off `MODE`.

- **`'fast'`** — small subsets, a few epochs. The whole notebook runs in a couple of minutes so you can
  *watch it work* and check the pipeline end-to-end. Numbers are illustrative, not final.
- **`'full'`** — the real thing: full CIFAR-10 (~92%), and **full ImageNet-32 (1.28M images)** which takes
  **hours**. Flip to this when you're ready to produce the actual results; the dashboard (below) tracks the
  long runs with a live ETA.

Leave it on `'fast'` to explore; switch to `'full'` for the capstone run.

In [ ]:
# ============================ THE ONE KNOB ============================
MODE = 'fast'            # 'fast' = minutes, watch it work   |   'full' = real results, hours
# =====================================================================

# Every stage reads its sizes from here, so 'fast' vs 'full' is a single edit.  In 'fast' we shrink
# BOTH the dataset (a random subset) AND the epoch count; in 'full' we use everything.
CONFIG = {
    'fast': dict(
        cifar_epochs   = 3,        # CIFAR-10/100: epochs
        cifar_subset   = 8_000,    #   ...on this many training images (None = all 50k)
        in32_epochs    = 2,        # ImageNet-32: epochs
        in32_subset    = 30_000,   #   ...on this many of the 1.28M images (a pipeline check, not the crossover)
    ),
    'full': dict(
        cifar_epochs   = 24,       # ~92% ResNet-18 on CIFAR-10
        cifar_subset   = None,     #   all 50k
        in32_epochs    = 40,       # the real ImageNet-32 schedule
        in32_subset    = None,     #   all 1,281,167 images  (this is the multi-hour run)
    ),
}[MODE]

print(f"MODE = {MODE!r}")
for k, v in CONFIG.items():
    print(f"  {k:14s} = {v}")
if MODE == 'fast':
    print("\n  -> pipeline/illustration run. Flip MODE='full' for real accuracy + the true crossover.")

## 1. From `sklearn.fit()` to a `Solver` to the loop

In **501** you called `model.fit()` and never saw the loop. In **502** you *wrote* it — `Solver._step()`
by hand. PyTorch is that same loop once more, and it's the one piece PyTorch still makes you write — so
it's where every "make it fast" trick below has to live. (`cifar10_train.ipynb` walks the full mapping.)

| CSED 501 | CSED 502, by hand | PyTorch (here) |
|---|---|---|
| `sklearn` did it | `softmax_loss` (`layers.py`) | `nn.CrossEntropyLoss` |
| `sklearn` did it | `ThreeLayerConvNet` (`cnn.py`) | `torchvision.models.resnet18` |
| `model.fit()` | `Solver._step()` (`solver.py`) | the `for x, y in loader:` loop |
| `sklearn` did it | `Solver._save_checkpoint()` | `torch.save(model.state_dict())` |

In [ ]:
# ---- Imports and path setup -------------------------------------------------------------------
# The repo splits reusable code into sibling folders; add them to the import path so this notebook
# can pull in the shared machinery instead of copy-pasting it:
#   ../common          -> gpu_check.py  (device detection; the grown-up cuda_check.py from 503)
#   ../a1-imagenet32   -> models.py     (resnet18/vit builders) and data.py (GPU-resident ImageNet-32)
#   ../perf            -> perfkit.py    (the cost estimator)
import os, sys, time, math
for rel in ('../common', '../a1-imagenet32', '../perf'):
    p = os.path.normpath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np                      # arrays / RNG
import torch                            # tensors + autograd + GPU
import torch.nn as nn                   # layers, losses
import matplotlib.pyplot as plt         # every graph in this notebook
from tqdm.auto import tqdm              # per-epoch progress bars (the within-epoch part of the dashboard)

from gpu_check import get_device, set_seed, enable_fast_matmul
import models as M                      # M.build('resnet18'|'vit'|'resnet50'|'vit_base', num_classes)

set_seed(42)                            # seed torch + numpy + python so runs are reproducible

## 2. One device, four platforms

502's GPU cell was three lines and broke the instant a teammate opened it on a **Mac** (Apple **MPS**,
not CUDA), on **Colab** (a different GPU each session), or on a **CPU** laptop. `get_device()` is what
`csed503/cuda_check.py` grew into: it prefers **CUDA → MPS → CPU**, *smoke-tests* MPS (some macOS builds
advertise it and then crash on the first tensor op), and on a multi-GPU box picks the best same-architecture
set. Same notebook, four platforms.

In [ ]:
# Pick the best device available, and choose a mixed-precision (AMP) dtype that this hardware likes.
DEVICE = get_device()                   # torch.device: 'cuda' here, 'mps' on a Mac, 'cpu' otherwise
if DEVICE.type == 'cuda':
    enable_fast_matmul()                # turn on TF32 matmuls + cuDNN autotuner (a free speedup on CUDA)

def pick_amp(device):
    """Which low-precision dtype should autocast use, and do we need a gradient loss-scaler?

    - bf16 has fp32's exponent range, so it needs NO loss scaler. Blackwell GPUs and Zen4 CPUs have it.
    - fp16 has a tiny range, so tiny gradients underflow to 0 -> a GradScaler multiplies the loss up
      before backward and divides it back out. Older CUDA cards (e.g. a Colab T4) only do fp16.
    - MPS autocast is patchy, so we skip AMP there and train in fp32.
    Returns (amp_dtype_or_None, use_scaler)."""
    if device.type == 'cuda':
        return (torch.bfloat16, False) if torch.cuda.is_bf16_supported() else (torch.float16, True)
    if device.type == 'cpu':
        return torch.bfloat16, False    # Zen4 AVX-512-BF16; a no-op speedup-wise on older CPUs but safe
    return None, False                  # MPS -> plain fp32

AMP_DTYPE, USE_SCALER = pick_amp(DEVICE)
print(f'Device {DEVICE} | AMP dtype {AMP_DTYPE} | loss scaler {USE_SCALER}')

## 3. The first wall — *feeding* the GPU

502 handed data to the model with a `DataLoader` and CPU worker processes. Right tool for big images —
it broke us twice at 32×32:

- **Spawn.** On Windows/macOS, DataLoader workers are *spawned* (fresh Python processes) and must
  **re-import** the `Dataset` class. A class defined in a notebook cell lives in `__main__` and can't be
  re-imported — the workers die with `Can't get attribute '...'`. So multi-worker loading was
  Linux/Colab-only; everyone else fell back to `num_workers=0` = **one CPU core** feeding the GPU.
- **The GPU was faster than one core could feed it.** At 32×32 a ResNet-18 trains faster than CPU
  augmentation produces batches, so the card sat *waiting on the CPU*.

The rewrite: **stop using the CPU for data.** CIFAR-10 is ~180 MB — hold it all **on the GPU** and
crop/flip/normalize there. Let's measure the difference on *this* machine.

In [ ]:
# ---- Load CIFAR-10 once, as tensors that live on the GPU -----------------------------------------
from torchvision.datasets import CIFAR10
from gpu_data import GPUImageLoader          # our on-device loader (src/a1-cv/gpu_data.py)

# torchvision downloads CIFAR-10 as uint8 arrays shaped (N, 32, 32, 3) = NHWC. Published channel stats:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
_root = os.path.join(os.getcwd(), 'data')
_tr = CIFAR10(_root, train=True,  download=True)
_te = CIFAR10(_root, train=False, download=True)

def to_device(ds):
    """(N,32,32,3) uint8 numpy -> a uint8 tensor on the GPU, in NCHW layout PyTorch convs expect."""
    x = torch.from_numpy(ds.data).to(DEVICE)          # copy the raw bytes to VRAM, still uint8 (cheap)
    x = x.permute(0, 3, 1, 2).contiguous()            # NHWC -> NCHW, and make it contiguous in memory
    y = torch.tensor(ds.targets, device=DEVICE)       # labels alongside the images
    return x, y

xtr, ytr = to_device(_tr)                             # ~150 MB of images, now resident on the card
xte, yte = to_device(_te)
print(f'train {tuple(xtr.shape)}  test {tuple(xte.shape)}  — all on {DEVICE}')

In [ ]:
# ---- Measure: CPU DataLoader (workers=0) vs the on-device loader --------------------------------
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class _CPUAug(Dataset):
    """A perfectly ordinary torchvision pipeline: one image at a time, augmented on the CPU."""
    def __init__(self, imgs, labels):
        self.imgs, self.labels = imgs, labels
        self.tf = transforms.Compose([                     # the exact CIFAR augmentation, PIL-based
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD)])
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return self.tf(Image.fromarray(self.imgs[i])), int(self.labels[i])

def feed_rate(loader, n_batches=20):
    """How many images/sec can this loader hand the model? (forward only, to isolate DATA delivery.)"""
    model = M.build('resnet18', 10).to(DEVICE).eval()
    seen, it = 0, iter(loader)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()     # make sure the GPU is idle before timing
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_batches):
            try:    x, y = next(it)
            except StopIteration: it = iter(loader); x, y = next(it)
            model(x.to(DEVICE, non_blocking=True)); seen += x.size(0)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()     # wait for the GPU to actually finish
    return seen / (time.time() - t0)

cpu_loader = DataLoader(_CPUAug(_tr.data, _tr.targets), batch_size=512, shuffle=True, num_workers=0)
gpu_loader = GPUImageLoader(xtr, ytr, 512, MEAN, STD, train=True, seed=42)
rates = {'CPU DataLoader (workers=0)': feed_rate(cpu_loader),
         'GPU-resident (on-device)':   feed_rate(gpu_loader)}
for k, v in rates.items(): print(f'{k:28s} {v:9,.0f} img/s')

fig, ax = plt.subplots(figsize=(5.5, 3.4))                  # <-- graph #1
ax.bar(list(rates), list(rates.values()), color=['#c44', '#4a8'])
ax.set_ylabel('images / sec delivered'); ax.set_title('Feeding the GPU at 32×32')
ax.bar_label(ax.containers[0], fmt='%.0f'); plt.xticks(rotation=8); plt.tight_layout(); plt.show()

## 4. Where PyTorch fought back — flips and non-contiguous views

Moving augmentation onto the GPU sounds simple. It wasn't. Two rewrites we didn't see coming.

### (a) The horizontal flip that stalled the pipeline
The obvious way to flip a random half of a batch is a boolean-mask write, `x[flip] = x[flip].flip(-1)`.
But `x[flip]` is *advanced indexing*: PyTorch calls `nonzero()` on the mask to find the selected rows,
and `nonzero()` is a **device→host sync** — the GPU stops and reports *how many* rows matched. Two syncs
per batch, ~2,500 batches/epoch, and it silently **breaks CUDA-graph capture**. The branch-free,
bitwise-identical fix is `torch.where`. Let's time both.

In [ ]:
def time_flip(fn, iters=400):
    """Ops/sec for one flip strategy. We time the op in isolation, syncing the GPU around the loop."""
    x    = torch.randn(512, 3, 32, 32, device=DEVICE)      # a batch of the right shape
    flip = torch.rand(512, device=DEVICE) < 0.5            # a random True/False per image
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters): fn(x, flip)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    return iters / (time.time() - t0)

def mask_write(x, flip):
    x2 = x.clone()                                         # (clone so repeated calls start clean)
    x2[flip] = x2[flip].flip(-1)                           # advanced index -> nonzero() -> HOST SYNC
    return x2
def where_flip(x, flip):
    return torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)   # pure elementwise select: no sync

a, b = time_flip(mask_write), time_flip(where_flip)
print(f'x[flip]=x[flip].flip(-1) : {a:8,.0f} flips/s   (nonzero -> host sync, breaks CUDA graphs)')
print(f'torch.where(...)         : {b:8,.0f} flips/s   ({b/a:.1f}x, and graph-safe)')

### (b) `.view()` on a non-contiguous tensor — the Mac-only crash
The per-sample random crop gathers pixels with fancy indexing, which returns a **non-contiguous** tensor
(its memory layout no longer matches its shape). CUDA usually tolerates that; **Apple MPS** does not — the
ViT's backward pass calls `.view()` on exactly such a tensor and **throws**. Two rules came out of that
afternoon, and they're now baked into the code:

- Use **`.reshape()`** (or an explicit `.contiguous()` first), never a bare **`.view()`**, on anything
  that's been transposed / indexed / permuted. `.view()` *requires* contiguity; `.reshape()` copies if it
  must.
- "Runs on every platform" is something you **debug into existence**, not a checkbox. Our crop returns a
  contiguous result on purpose; the flip is `torch.where`; nothing downstream assumes a memory layout.

## 5. The counterintuitive one — mixed precision × memory format

Mixed precision (**AMP**) was never in the coursework (503 only got it through `Trainer(fp16=True)`).
`torch.autocast` runs the matmuls in low precision — **but which** low precision, and **which memory
layout**, interact in a way that cost us days.

`channels_last` (NHWC) is cuDNN's preferred conv layout, so the folklore is "always use it." On this
hardware that folklore is **half wrong**, and the sign flips with the dtype. Don't trust us — measure it:

In [ ]:
def step_rate(amp_dtype, channels_last, iters=60, warmup=15):
    """Training-step throughput (fwd+bwd+opt) for one (dtype, memory-format) combination."""
    m = M.build('resnet18', 10).to(DEVICE)
    if channels_last:
        m = m.to(memory_format=torch.channels_last)        # store the weights in NHWC
    opt = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
    x = torch.randn(512, 3, 32, 32, device=DEVICE)
    y = torch.randint(0, 10, (512,), device=DEVICE)
    if channels_last:
        x = x.contiguous(memory_format=torch.channels_last) # and feed activations in NHWC too
    def step():
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            nn.functional.cross_entropy(m(x), y).backward()
        opt.step()
    for _ in range(warmup): step()                          # warm up cuDNN autotuning first
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters): step()
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    return 512 * iters / (time.time() - t0)

configs = [('fp32', None, False), ('fp16', torch.float16, False), ('fp16 + chan_last', torch.float16, True),
           ('bf16', torch.bfloat16, False), ('bf16 + chan_last', torch.bfloat16, True)]
rate = {name: step_rate(dt, cl) for name, dt, cl in configs}
base = rate['fp32']
for name in rate: print(f'{name:18s} {rate[name]:9,.0f} img/s   {rate[name]/base:4.2f}x vs fp32')

fig, ax = plt.subplots(figsize=(7, 3.6))                    # <-- graph #2
ax.bar(list(rate), list(rate.values()), color=['#888', '#69c', '#c44', '#48a', '#2a6'])
ax.set_ylabel('img/s (resnet18, bs512)'); ax.set_title('Mixed precision × memory format — same GPU')
ax.bar_label(ax.containers[0], fmt='%.0f'); plt.xticks(rotation=12); plt.tight_layout(); plt.show()

Read the bars. **`channels_last` *helps* under bf16 and *hurts* under fp16** — on the RTX PRO 6000 we
measured bf16+`channels_last` as the fastest config *and* fp16+`channels_last` as roughly **4× slower** than
plain fp16 (a pathological cuDNN NHWC-fp16 path). Same flag, opposite sign, decided by the dtype. There is
no substitute for measuring on the target — which is exactly what pushes us toward a **factory** that
measures for us (§8).

The habit under all of it: before optimizing, know **what the GPU is waiting on** —

| symptom | you are… | the fix |
|---|---|---|
| low GPU util, one CPU core pegged | **CPU-bound** (data) | more workers, or on-device data (§3) |
| low GPU util, low CPU util | **launch-bound** (batch too small) | bigger batch; avoid per-batch host syncs (§4) |
| high GPU util | **compute-bound** | you're done — need a bigger model or a bigger card |

## 6. A real training run — with a live dashboard

Now we wire the fast path into one `train()` used for *every* run below (CIFAR here, ImageNet-32 in §9).
For the long runs (`MODE='full'`, ImageNet-32 = hours) staring at a scrolling log is useless, so `train()`
redraws a **dashboard after every epoch**: the loss and accuracy curves so far, plus a status line with
throughput and an **ETA**. Read the comments — this is `Solver.train()`, one rung up, instrumented.

In [ ]:
from IPython.display import clear_output

@torch.no_grad()                                           # evaluation never needs gradients
def evaluate(model, val_batches, channels_last):
    """Top-1 accuracy over a validation pass. val_batches() returns a fresh iterator of (x, y)."""
    model.eval()                                           # BatchNorm/Dropout into inference mode
    correct = torch.zeros((), device=DEVICE); total = 0    # accumulate on the GPU — no per-batch sync
    for x, y in val_batches():
        if channels_last: x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            preds = model(x).argmax(1)                     # class with the highest logit
        correct += (preds == y).sum(); total += y.size(0)
    model.train()                                          # back to training mode
    return (correct / total).item()                        # one GPU->host sync for the whole pass

def dashboard(title, hist, epoch, epochs, t_start):
    """Redraw the live progress panel: status line (with ETA) + loss/accuracy curves."""
    clear_output(wait=True)                                # replace the previous frame in-place
    elapsed = time.time() - t_start
    eta = elapsed / epoch * (epochs - epoch)               # simple linear ETA from mean epoch time
    def hms(s): return f'{int(s//3600)}h{int(s%3600//60):02d}m{int(s%60):02d}s'
    print(f'[{title}]  epoch {epoch}/{epochs}   '
          f'val top-1 {hist["top1"][-1]:6.2%}   loss {hist["loss"][-1]:.3f}   '
          f'{hist["img_s"][-1]:8,.0f} img/s   elapsed {hms(elapsed)}   ETA {hms(eta)}')
    xs = range(1, epoch + 1)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
    a1.plot(xs, hist['loss'], color='#c44'); a1.set_title('train loss'); a1.set_xlabel('epoch')
    a2.plot(xs, hist['top1'], color='#2a6'); a2.set_title('val top-1'); a2.set_xlabel('epoch')
    a2.set_ylim(0, 1); a2.grid(alpha=.3)
    plt.tight_layout(); plt.show(); plt.close(fig)         # close so 40 epochs don't leak 40 figures

def train(title, model, train_batches, val_batches, n_train, epochs, *, lr, opt='sgd',
          strong_aug=None, live=True):
    """Solver.train(), instrumented. `train_batches()`/`val_batches()` each return a fresh (x,y) iterator
    for one pass; `n_train` is images/epoch (for img/s + the dashboard tqdm total). `strong_aug` is the
    ViT's mixup/cutmix+erasing callback (None for the CNN). Returns a history dict."""
    model = model.to(DEVICE)
    CHAN_LAST = (DEVICE.type == 'cuda' and AMP_DTYPE is torch.bfloat16)   # the measured rule from §5
    if CHAN_LAST:
        model = model.to(memory_format=torch.channels_last)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)        # label smoothing: a mild, standard regularizer

    # Optimizer + schedule follow the ARCHITECTURE (502/503 lesson): SGD+momentum for the CNN, AdamW for
    # the transformer. 5-epoch linear warmup -> cosine decay is the modern default for both.
    opt_obj = (torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05) if opt == 'adamw'
               else torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=5e-4))
    warm_ep = min(5, max(1, epochs - 1))
    warm = torch.optim.lr_scheduler.LinearLR(opt_obj, 0.1, total_iters=warm_ep)
    cos  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_obj, T_max=max(1, epochs - warm_ep))
    sched = torch.optim.lr_scheduler.SequentialLR(opt_obj, [warm, cos], [warm_ep])
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_SCALER)        # only active for fp16

    hist = {'loss': [], 'top1': [], 'img_s': []}
    t_start = time.time()
    steps = math.ceil(n_train / 512)
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); seen = 0
        # >>> THE optimization §4 was really about, applied to the loop itself. <<<
        # Accumulate the running loss ON THE GPU and sync exactly ONCE, after the epoch. Writing
        # `loss.item()` inside the batch loop forces a GPU->host sync every single step: the CPU then
        # can't launch the next batch until the GPU stops and reports the loss, so a small 32x32 model
        # stalls between steps and NEITHER the GPU nor the CPU is stressed (~90% -> ~30% util). This is
        # the exact host-sync trap as the flip's nonzero() in §4 — just hiding in the metrics line.
        loss_sum = torch.zeros((), device=DEVICE)
        for x, y in tqdm(train_batches(), total=steps, desc=f'{title} ep {ep}/{epochs}', leave=False):
            if CHAN_LAST: x = x.contiguous(memory_format=torch.channels_last)
            opt_obj.zero_grad(set_to_none=True)            # clear last step's grads (set_to_none = faster)
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
                if strong_aug is not None:                 # ViT: mixup/CutMix -> two soft targets
                    x, ya, yb, lam = strong_aug(x, y)
                    out = model(x); loss = lam * crit(out, ya) + (1 - lam) * crit(out, yb)
                else:                                      # CNN: plain cross-entropy
                    loss = crit(model(x), y)
            # Backward + step. With fp16 the scaler prevents gradient underflow; with bf16 it's a no-op.
            if USE_SCALER: scaler.scale(loss).backward(); scaler.step(opt_obj); scaler.update()
            else:          loss.backward(); opt_obj.step()
            loss_sum += loss.detach() * y.size(0)          # stays on the GPU — the CPU never waits
            seen += y.size(0)
        sched.step()                                       # one LR step per epoch
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        hist['loss'].append((loss_sum / seen).item())      # <-- the single GPU->host sync for the epoch
        hist['top1'].append(evaluate(model, val_batches, CHAN_LAST))
        hist['img_s'].append(seen / (time.time() - t0))
        if live: dashboard(title, hist, ep, epochs, t_start)
    return hist

### Train ResNet-18 on CIFAR-10
On the workstation GPU `MODE='full'` reaches the low-90s% in ~2 minutes; `MODE='fast'` just proves the
pipeline. The `GPUImageLoader` yields on-device, augmented batches — so `train_batches()` is just "make a
fresh loader pass."

In [ ]:
# Build the on-device loaders (optionally on a random subset in 'fast' mode).
def cifar_loaders(subset):
    xt, yt = (xtr, ytr) if subset is None else (xtr[:subset], ytr[:subset])
    train_loader = GPUImageLoader(xt, yt, 512, MEAN, STD, train=True, seed=42)
    test_loader  = GPUImageLoader(xte, yte, 512, MEAN, STD, train=False)
    return train_loader, test_loader, len(yt)

c_train, c_test, c_n = cifar_loaders(CONFIG['cifar_subset'])
resnet_hist = train('ResNet-18 / CIFAR-10',
                    M.build('resnet18', num_classes=10),
                    train_batches=lambda: iter(c_train),   # a fresh shuffled+augmented pass
                    val_batches=lambda: iter(c_test),
                    n_train=c_n, epochs=CONFIG['cifar_epochs'],
                    lr=0.1 * 512 / 256)                     # SGD LR scaled to batch size (502 lesson)
print(f'ResNet-18 best val top-1: {max(resnet_hist["top1"]):.2%}')

## 7. The same job, a Transformer — new recipe, same fast loop

A **ViT** is the attention we hand-built in 502/503, now over image *patches*. It ports with one argument
(`num_classes`), but its **training recipe does not**: hand it the ResNet's recipe (SGD, `lr=0.1`) and it
lands in the 50s. The fixes — **AdamW**, LR **warmup**, and **strong augmentation** (mixup/CutMix + random
erasing) — are the transformer's *substitute for the inductive bias it doesn't have*. On 50k images the CNN
still wins; hold that thought until §9.

In [ ]:
# The ViT's strong augmentation lives in a1-imagenet32/data.py (GPU-side mixup/CutMix + erasing).
from data import mixup_cutmix, random_erasing_

def vit_aug(x, y):
    """Erase a random patch, then mixup/CutMix with another image in the batch -> two soft targets."""
    x = random_erasing_(x)                                 # cut a random hole in ~25% of the batch
    return mixup_cutmix(x, y)                              # returns (x, y_a, y_b, lam) for a mixed loss

vit_hist = train('ViT / CIFAR-10',
                 M.build('vit', num_classes=10),
                 train_batches=lambda: iter(c_train), val_batches=lambda: iter(c_test),
                 n_train=c_n, epochs=CONFIG['cifar_epochs'],
                 lr=1e-3, opt='adamw', strong_aug=vit_aug) # AdamW + strong aug = the ViT recipe
print(f'ViT best {max(vit_hist["top1"]):.2%}   vs ResNet-18 {max(resnet_hist["top1"]):.2%}  '
      f'-> the CNN wins on 50k images')

fig, ax = plt.subplots(figsize=(6.5, 4))                   # <-- graph #3: the two curves together
ax.plot(range(1, len(resnet_hist['top1'])+1), resnet_hist['top1'], label='ResNet-18 (CNN)', color='#2a6')
ax.plot(range(1, len(vit_hist['top1'])+1),    vit_hist['top1'],    label='ViT (Transformer)', color='#96c')
ax.set_xlabel('epoch'); ax.set_ylabel('val top-1'); ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f'CNN vs Transformer — CIFAR-10 ({c_n:,} images, MODE={MODE})'); plt.tight_layout(); plt.show()

## 8. Benchmark small, **predict** big — will ImageNet-32 take 1 hour or 1 day?

Before launching a run that might take all day, predict it. The key fact: **ImageNet-32 uses the same
models at the same 32×32 resolution as CIFAR** — so the *per-image* compute cost is (almost) identical.
That means we can **benchmark on CIFAR (seconds) and extrapolate to ImageNet-32's 1.28M images**.

We do it two ways and compare them in §9:
1. **Naive throughput extrapolation** — measure steady-state `img/s` on CIFAR, then `time ≈ images × epochs / img_s`. Intuitive, and *slightly optimistic* (we'll see why).
2. **`perfkit`'s roofline** (`../perf/`, the grown-up `hyper_sweep.py`) — counts each model's FLOPs
   *including the 1000-class head ImageNet needs* and divides by the GPU's measured throughput, returning a
   `[fast, slow]` band. This is the factory's redesign gate: *"minutes or days?"* answered with no run.

In [ ]:
# ---- (1) Naive: measure steady-state img/s on CIFAR, extrapolate by image count ------------------
def measure_img_s(model, loader, n_train, warmup_batches=10, measure_batches=40):
    """Steady-state training throughput: skip warmup batches (epoch-1 cuDNN autotune), then time some."""
    model = model.to(DEVICE)
    CHAN = (DEVICE.type == 'cuda' and AMP_DTYPE is torch.bfloat16)
    if CHAN: model = model.to(memory_format=torch.channels_last)
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    it = iter(loader); seen = 0
    def one():
        nonlocal it
        try: x, y = next(it)
        except StopIteration: it = iter(loader); x, y = next(it)
        if CHAN: x = x.contiguous(memory_format=torch.channels_last)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            crit(model(x), y).backward()
        opt.step(); return x.size(0)
    for _ in range(warmup_batches): one()
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(measure_batches): seen += one()
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    return seen / (time.time() - t0)

# A SECOND benchmark dataset: CIFAR-100 (same 32x32, 100 classes). We check that the per-image cost is
# STABLE across datasets before trusting it to predict a THIRD one (ImageNet-32, 1000 classes).
from torchvision.datasets import CIFAR100
_c100 = CIFAR100(_root, train=True, download=True)
x100 = torch.from_numpy(_c100.data).to(DEVICE).permute(0, 3, 1, 2).contiguous()
y100 = torch.tensor(_c100.targets, device=DEVICE)
if CONFIG['cifar_subset'] is not None:
    x100, y100 = x100[:CONFIG['cifar_subset']], y100[:CONFIG['cifar_subset']]
c100_train = GPUImageLoader(x100, y100, 512, MEAN, STD, train=True, seed=42)   # norm consts don't affect timing

IN32_N, IN32_EPOCHS = 1_281_167, CONFIG['in32_epochs']       # ImageNet-32 full size × our schedule
bench = {}
for name in ('resnet18', 'vit'):
    r10  = measure_img_s(M.build(name, 10),  c_train,    c_n)        # benchmark on CIFAR-10  (10 classes)
    r100 = measure_img_s(M.build(name, 100), c100_train, len(y100))  # benchmark on CIFAR-100 (100 classes)
    r = 0.5 * (r10 + r100)                                   # ~equal -> one stable per-image rate to extrapolate
    naive_hours = IN32_N * IN32_EPOCHS / r / 3600            # images*epochs / (img/s) -> hours
    bench[name] = dict(img_s=r, naive_hours=naive_hours)
    print(f'{name:9s}: CIFAR-10 {r10:8,.0f}  |  CIFAR-100 {r100:8,.0f} img/s  (consistent)  ->  '
          f'naive ImageNet-32 ({IN32_N:,}×{IN32_EPOCHS} ep): {naive_hours:5.2f} h')

In [ ]:
# ---- (2) perfkit roofline, WITH the ImageNet 1000-class head -------------------------------------
import perfkit as pk
peak = pk.probe_matmul_tflops(DEVICE)                        # quick burst-GEMM probe (a few seconds)
p_tflops = peak.get('bf16') or peak.get('tf32') or peak.get('fp32')
print(f'GPU peak ~{p_tflops:.0f} TFLOPS\n')

for name in ('resnet18', 'vit'):
    # count_flops_per_image builds the model with num_classes=1000 (ImageNet) so the classifier head,
    # which the CIFAR benchmark under-counts, is included.
    flops = pk.count_flops_per_image(lambda name=name: M.build(name, 1000))
    work  = pk.workload_spec(name, n_train=IN32_N, n_val=50_000, batch_size=512,
                             epochs=IN32_EPOCHS, flops=flops)
    est = pk.tier1_estimate(work, p_tflops=p_tflops)        # [optimistic, pessimistic] over an MFU band
    bench[name]['roofline_h'] = (est['optimistic']['total_s'] / 3600, est['pessimistic']['total_s'] / 3600)
    print(f'{name:9s}: roofline predict ImageNet-32 = '
          f'{bench[name]["roofline_h"][0]:.2f} – {bench[name]["roofline_h"][1]:.2f} h   '
          f'(naive was {bench[name]["naive_hours"]:.2f} h)')
print('\nTwo predictions on the table. §9 runs it and checks who was right.')

## 9. ImageNet-32 — where the CNN breaks down, and predicted vs **actual**

Now the payoff, and the reason for every optimization above. We train the *same two models* on
**ImageNet-32** — 1,281,167 images, 1000 classes, still 32×32 — using the GPU-resident loader from
`a1-imagenet32/data.py` (the whole 3.9 GB dataset lives in VRAM; there is no DataLoader).

Two things to watch:

- **The crossover.** On CIFAR's 50k the CNN won easily. Here, with **25× the data**, the ViT is no longer
  starved — it *learns* the spatial structure the CNN gets for free, and closes or beats the gap. In the
  full study the from-scratch **ViT overtakes** (val top-1 **42.76%** vs the ResNet's **41.54%**). That is
  the whole thesis: *CNNs win on small data, Transformers win once the data is there.* Why not always use
  the faster-to-train CNN? Because its inductive bias is also a **ceiling** — the ViT keeps improving with
  data past where the CNN plateaus.
- **Predicted vs actual.** We timed §8's predictions; now we measure the real thing and see how close we
  got — and what to fix.

> **`MODE='fast'` runs a ~30k-image subset for a couple of epochs** — enough to prove the pipeline and the
> prediction *method*, **not** enough to show the real crossover (that needs the full 1.28M, i.e.
> `MODE='full'`, hours). The notebook says so honestly below.

In [ ]:
from data import GpuImageNet32                          # the GPU-resident ImageNet-32 loader (§data.py)

# Load ImageNet-32 into VRAM. subset=None in 'full' loads all 1.28M images (~3.9 GB); 'fast' takes a
# RANDOM subset across all 1000 classes (the file is class-sorted, so a sequential slice would be a
# handful of classes — data.py handles the random sampling for us).
in32_train = GpuImageNet32(DEVICE, split='train', subset=CONFIG['in32_subset'], seed=0)
in32_val   = GpuImageNet32(DEVICE, split='val',   subset=None)
print(f"ImageNet-32 resident on {DEVICE}: {in32_train.gb():.1f} GB  |  "
      f"{in32_train.n:,} train, {in32_val.n:,} val, {in32_train.n_classes} classes")

# GpuImageNet32.epoch(bs, train) yields augmented (train) or plain (val) batches — same (x,y) contract
# the train() loop expects, so we just wrap it in the batches() lambdas.
BS = 512
in32_batches     = lambda: in32_train.epoch(BS, train=True)
in32_val_batches = lambda: in32_val.epoch(BS, train=False)

In [ ]:
# --- Train ResNet-18 on ImageNet-32, and TIME it (for predicted-vs-actual) ---
t0 = time.time()
in32_resnet = train('ResNet-18 / ImageNet-32',
                    M.build('resnet18', num_classes=in32_train.n_classes),
                    in32_batches, in32_val_batches,
                    n_train=in32_train.n, epochs=IN32_EPOCHS, lr=0.1 * 512 / 256)
resnet_actual_h = (time.time() - t0) / 3600
print(f'ResNet-18 ImageNet-32 best val top-1: {max(in32_resnet["top1"]):.2%}   '
      f'(actual wall-clock {resnet_actual_h*60:.1f} min)')

In [ ]:
# --- Train the ViT on ImageNet-32 (strong aug), and TIME it ---
t0 = time.time()
in32_vit = train('ViT / ImageNet-32',
                 M.build('vit', num_classes=in32_train.n_classes),
                 in32_batches, in32_val_batches,
                 n_train=in32_train.n, epochs=IN32_EPOCHS, lr=1e-3, opt='adamw', strong_aug=vit_aug)
vit_actual_h = (time.time() - t0) / 3600
print(f'ViT ImageNet-32 best val top-1: {max(in32_vit["top1"]):.2%}   '
      f'(actual wall-clock {vit_actual_h*60:.1f} min)')

In [ ]:
# --- The crossover, side by side (CIFAR vs ImageNet-32) --------------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))        # <-- graph #4
a1.plot(range(1, len(resnet_hist['top1'])+1), resnet_hist['top1'], color='#2a6', label='ResNet-18')
a1.plot(range(1, len(vit_hist['top1'])+1),    vit_hist['top1'],    color='#96c', label='ViT')
a1.set_title(f'CIFAR-10 ({c_n:,} imgs) — CNN ahead'); a1.set_xlabel('epoch'); a1.set_ylim(0,1)
a1.legend(); a1.grid(alpha=.3)
a2.plot(range(1, len(in32_resnet['top1'])+1), in32_resnet['top1'], color='#2a6', label='ResNet-18')
a2.plot(range(1, len(in32_vit['top1'])+1),    in32_vit['top1'],    color='#96c', label='ViT')
a2.set_title(f'ImageNet-32 ({in32_train.n:,} imgs) — the gap closes'); a2.set_xlabel('epoch'); a2.set_ylim(0,1)
a2.legend(); a2.grid(alpha=.3); plt.tight_layout(); plt.show()
if MODE == 'fast':
    print("NOTE: MODE='fast' — subset/short. The real crossover (ViT 42.76% > CNN 41.54%) needs MODE='full'.")

In [ ]:
# --- Predicted vs ACTUAL: how good were the §8 estimates, and what's the error? --------------------
# In 'fast' mode we predicted the FULL 1.28M run but only trained a subset for a few epochs, so we scale
# the actual to a per-(image*epoch) basis to compare methods apples-to-apples.
def to_full_hours(actual_h):
    ran = in32_train.n * IN32_EPOCHS                        # image*epochs we actually did
    full = 1_281_167 * IN32_EPOCHS
    return actual_h * full / ran                            # linear scale-up (dominated by the inner loop)

print(f"{'model':9s} {'naive':>8s} {'roofline':>16s} {'ACTUAL':>10s}   (hours, 40-epoch full run)")
for name, actual_h in [('resnet18', resnet_actual_h), ('vit', vit_actual_h)]:
    b = bench[name]; act = to_full_hours(actual_h)
    lo, hi = b['roofline_h']
    err = (b['naive_hours'] - act) / act * 100
    print(f"{name:9s} {b['naive_hours']:7.2f}h {lo:6.2f}-{hi:5.2f}h {act:9.2f}h   naive off {err:+.0f}%")
print("""
Reading it:  the NAIVE img/s extrapolation tends to UNDER-predict, and the fixes are the lesson —
  * ImageNet has a 1000-class head (CIFAR's was 10): more FLOPs per step the CIFAR benchmark never saw.
    perfkit's roofline counts them, which is why its band brackets the truth better.
  * per-epoch eval on 50k val images, and epoch-1 cuDNN autotune, are real wall-clock the naive
    images*epochs/img_s ignores.
  * the ViT's strong augmentation (mixup/CutMix/erasing) costs per-batch time the CIFAR CNN benchmark
    didn't include.
The fix is the factory principle: **benchmark the SAME model + head + augmentation you'll run**, or use a
FLOPs-aware estimate — then the prediction lands, and you can schedule a day of runs with confidence.""")

## 10. The model factory

Stack the pieces and you have the backbone `hyper_sweep.py` was reaching for:

- **runs anywhere** — one `get_device()`, four platforms (§2), debugged through the MPS/spawn/`.view()`
  scars (§3–4);
- **fast** — on-device data, AMP, the measured `channels_last`-by-dtype rule, the bound-diagnosis habit (§3–5);
- **both families, at any scale** — CNN *and* Transformer, from 50k to 1.28M images, on one loop (§6–9);
- **predictable** — cost estimated before the run, then checked against reality and corrected (§8–9).

A **model factory** points that at a search space — architecture family, width/depth, recipe — and lets it
*estimate → schedule → run* across whatever hardware is on hand (the two RTX PRO 6000s here, a Mac
overnight, a Colab tab), keeping the accuracy/cost Pareto front, and choosing **CNN or Transformer by the
data budget** it's given. The **what to search** is the crossover we just watched; the **how to run it
cheaply, everywhere, and know the bill in advance** is everything above.

We started at `sklearn.fit()` and a hand-written `Solver`. This is where that goes.